In [1]:
# !pip uninstall -y pytabkit torch torchvision torchaudio -q
# !pip install torch torchvision torchaudio \
#     --index-url https://download.pytorch.org/whl/cu121 -q
# !pip install pytabkit -q
# !pip install optbinning

In [22]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import os

from optbinning import OptimalBinning
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

## Config

In [13]:
MAX_BIN = 5
MIN_BIN_SIZE = 0.05

ROOT_PATH = "input_data/row_fs_data"
TARGET = 'TARGET'
ID = 'SK_ID_CURR'

## Reading Train | Test

In [4]:
app_train = pd.read_csv(os.path.join(ROOT_PATH, "app_train_fs.csv"))
app_test = pd.read_csv(os.path.join(ROOT_PATH, "app_test_fs.csv"))

print(f"Shape of app_train : {app_train.shape} | Shape of app_test  :  {app_test.shape}")
print(f"Bad rate           : {app_train[TARGET].value_counts(normalize=True)[1].round(2) * 100} %")

Shape of app_train : (307511, 309) | Shape of app_test  :  (48744, 308)
Bad rate           : 8.0 %


## WoE / IV (Optbinning)

In [20]:
feat_cols = [col for col in app_train.columns if col not in [TARGET, ID]]
iv_report = []

train_woe_report = {
    ID: app_train[ID],
    TARGET: app_train[TARGET]
}

test_woe_report = {
    ID: app_test[ID]
}

for col in feat_cols:

    feat = app_train[col]
    target = app_train[TARGET]

    opt = OptimalBinning(
        name=col,
        dtype="numerical" if pd.api.types.is_numeric_dtype(feat) else "categorical",
        max_n_bins=MAX_BIN,
        min_bin_size=MIN_BIN_SIZE
    )

    opt.fit(feat, target)

    # IV Report
    bin_info = opt.binning_table.build()
    iv = bin_info["IV"].sum()

    iv_report.append({
        "feature": col,
        "iv": iv
    })

    # WoE transformed features
    train_woe_report[col] = opt.transform(feat, metric="woe")
    test_woe_report[col] = opt.transform(app_test[col], metric="woe")

app_train_woe = pd.DataFrame(train_woe_report)
app_test_woe = pd.DataFrame(test_woe_report)

iv_report = (
    pd.DataFrame(iv_report)
    .sort_values("iv", ascending=False)
)

print(app_train_woe.shape)
print(app_test_woe.shape)
print(iv_report.head(10))

(307511, 309)
(48744, 308)
                            feature        iv
1                      EXT_SOURCE_3  0.652011
0                      EXT_SOURCE_2  0.612655
2                      EXT_SOURCE_1  0.291974
4         DAYS_CREDIT_MEDIAN_bureau  0.246415
3           DAYS_CREDIT_MEAN_bureau  0.245909
5                     DAYS_EMPLOYED  0.219710
7  DAYS_CREDIT_UPDATE_MEDIAN_bureau  0.201131
6          AMT_PAYMENT_MIN_inst_pmt  0.189707
8    DAYS_CREDIT_UPDATE_MEAN_bureau  0.187330
9                        DAYS_BIRTH  0.168098


In [28]:
X, y = app_train_woe.drop([ID, TARGET], axis=1), app_train_woe[TARGET]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f"Scale Pos Weight : {scale_pos_weight}")

base_model = lgb.LGBMClassifier(
    objective="binary",
    metric="auc",
    boosting_type="gbdt",
    n_estimators=5000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

base_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100)
    ]
)

y_pred = base_model.predict_proba(
    X_valid,
    num_iteration=base_model.best_iteration_
)[:, 1]

auc_score = roc_auc_score(y_valid, y_pred)
gini_score = 2 * auc_score - 1

print("\n" + "=" * 50)
print(f"Best Iteration : {base_model.best_iteration_}")
print(f"AUC Score      : {auc_score:.6f}")
print(f"Gini Score     : {gini_score:.6f}")
print("=" * 50)

Scale Pos Weight : 11.38710976837865
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.773587
[200]	valid_0's auc: 0.777866
[300]	valid_0's auc: 0.778582
Early stopping, best iteration is:
[284]	valid_0's auc: 0.778714

Best Iteration : 284
AUC Score      : 0.778714
Gini Score     : 0.557428


In [30]:
feature_importance = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance": base_model.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\nTop 20 Features")
display(feature_importance.head(20))


Top 20 Features


,feature,importance
0,EXT_SOURCE_1,234
1,AMT_ANNUITY,212
2,EXT_SOURCE_2,194
3,EXT_SOURCE_3,155
4,AMT_GOODS_PRICE,145
5,OWN_CAR_AGE,137
6,AMT_PAYMENT_MIN_inst_pmt,125
7,ORGANIZATION_TYPE,124
8,AMT_CREDIT,113
9,DAYS_EMPLOYED,107
